# 02 - BP/RP Basis Reconstruction Comparison (No Smoothing)

This notebook compares basis families on raw (unsmoothed) spectra:
- keep BP and RP separate;
- compare chebyshev, legendre, bspline, fourier in wavelength space;
- evaluate reconstruction quality with a BP+RP combined decision metric;
- smoothing exploration is handled separately in notebook 04.

In [6]:
%matplotlib inline

from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
SCRIPT_PATH = ROOT / "02_generate_basis_features.py"
spec = spec_from_file_location("bp_basis_step02", SCRIPT_PATH)
step02 = module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = step02
spec.loader.exec_module(step02)

from _common import BP_SAMPLED_CSV, FEATURES_DIR, RESULTS_DIR, RP_SAMPLED_CSV, flatten_feature_blocks, l2_normalize

## Check that BP and RP sampled spectra exist

In [7]:
for path in [BP_SAMPLED_CSV, RP_SAMPLED_CSV]:
    print(path, "->", path.exists())

if not BP_SAMPLED_CSV.exists() or not RP_SAMPLED_CSV.exists():
    raise RuntimeError("Run 01_prepare_inputs.ipynb first to create separate BP/RP sampled spectra.")

/Users/erikak/Documents/uni/bakalauras/kodas/bp_rp_basis_experiment/data/bp_sampled_spectra.csv -> True
/Users/erikak/Documents/uni/bakalauras/kodas/bp_rp_basis_experiment/data/rp_sampled_spectra.csv -> True


## Load BP and RP blocks

In [8]:
bp = step02.load_block(BP_SAMPLED_CSV)
rp = step02.load_block(RP_SAMPLED_CSV)
step02.check_alignment(bp, rp)
preview_idx = step02.choose_example_indices(bp.labels, n_examples=4)

print("BP:", bp.flux.shape, bp.wavelengths[0], bp.wavelengths[-1])
print("RP:", rp.flux.shape, rp.wavelengths[0], rp.wavelengths[-1])
print("Preview source_ids:", bp.source_ids[preview_idx].tolist())

BP: (2815, 154) 336.0 642.0
RP: (2815, 193) 636.0 1020.0
Preview source_ids: [1792620565667968, 6052403489630720, 35928500943041152, 93468972376758144]


## Pick one star and compare raw vs. basis reconstructions across K

In [22]:
selected_source_id = int(bp.source_ids[preview_idx[0]])
basis_preview = ["chebyshev", "legendre", "bspline", "fourier"]
k_preview = [5, 10, 20, 40, 50]
smoothing_preview = "none"

fig = step02.plot_selected_star_reconstructions_interactive(
    bp,
    rp,
    source_id=selected_source_id,
    bases=basis_preview,
    k_list=k_preview,
    smoothing=smoothing_preview,
)
fig.show()

## Run the basis x K comparison grid (no smoothing)

In [17]:
bases = ["chebyshev", "legendre", "bspline", "fourier"]
smoothing = "none"
k_list = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

reconstruction_dir = RESULTS_DIR / "reconstruction"
reconstruction_dir.mkdir(parents=True, exist_ok=True)

metric_frames = []
preview_cache = {}
generated = []

for basis in bases:
    for K in k_list:
        print(f"basis={basis:10s} K={K:02d}")
        bp_fit = step02.build_block_fit(bp, basis, smoothing, K)
        rp_fit = step02.build_block_fit(rp, basis, smoothing, K)

        feat_df = flatten_feature_blocks(bp.source_ids, bp.labels, bp_fit.coeffs, rp_fit.coeffs)
        coeff_cols = [c for c in feat_df.columns if c.startswith("c")]
        feat_df = l2_normalize(feat_df, coeff_cols=coeff_cols)
        out_path = FEATURES_DIR / f"{basis}_{smoothing}_{K:02d}_L2.csv"
        feat_df.to_csv(out_path, index=False)
        generated.append(out_path.name)

        metric_frames.append(
            step02.metric_frame(
                bp, arm="bp", basis=basis, smoothing=smoothing,
                n_coeffs=K, smoothed_flux=bp_fit.smoothed_flux,
                reconstructed_flux=bp_fit.reconstructed_flux,
            )
        )
        metric_frames.append(
            step02.metric_frame(
                rp, arm="rp", basis=basis, smoothing=smoothing,
                n_coeffs=K, smoothed_flux=rp_fit.smoothed_flux,
                reconstructed_flux=rp_fit.reconstructed_flux,
            )
        )
        metric_frames.append(
            step02.combined_metric_frame(
                bp, rp, basis=basis, smoothing=smoothing, n_coeffs=K,
                bp_smoothed_flux=bp_fit.smoothed_flux,
                rp_smoothed_flux=rp_fit.smoothed_flux,
                bp_reconstructed_flux=bp_fit.reconstructed_flux,
                rp_reconstructed_flux=rp_fit.reconstructed_flux,
            )
        )

        preview_cache[(basis, smoothing, K, "bp")] = bp_fit.reconstructed_flux[preview_idx].copy()
        preview_cache[(basis, smoothing, K, "rp")] = rp_fit.reconstructed_flux[preview_idx].copy()

raw_metrics = pd.concat(metric_frames, ignore_index=True)
summary = step02.summarize_metric_frame(raw_metrics)

raw_metrics_path = reconstruction_dir / "reconstruction_metrics_by_star.csv"
summary_path = reconstruction_dir / "reconstruction_summary.csv"
raw_metrics.to_csv(raw_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
combined_ranking = step02.rank_combined_configs(summary)
combined_ranking_path = reconstruction_dir / "reconstruction_summary_combined_ranked.csv"
combined_ranking.to_csv(combined_ranking_path, index=False)

metric_plots = step02.plot_metric_curves(summary, reconstruction_dir)
best_bp_plot = step02.plot_best_reconstructions("bp", bp, summary, preview_cache, preview_idx, reconstruction_dir, top_n=3)
best_rp_plot = step02.plot_best_reconstructions("rp", rp, summary, preview_cache, preview_idx, reconstruction_dir, top_n=3)

print("Generated feature files:", len(generated))
print("Per-star metrics:", raw_metrics_path)
print("Summary:", summary_path)
print("Metric plots:", [p.name for p in metric_plots])
print("Combined ranking:", combined_ranking_path)
print("Best-example plots:", [p.name for p in [best_bp_plot, best_rp_plot] if p is not None])

basis=chebyshev  K=05
basis=chebyshev  K=10
basis=chebyshev  K=15
basis=chebyshev  K=20
basis=chebyshev  K=25
basis=chebyshev  K=30
basis=chebyshev  K=35
basis=chebyshev  K=40
basis=chebyshev  K=45
basis=chebyshev  K=50
basis=legendre   K=05
basis=legendre   K=10
basis=legendre   K=15
basis=legendre   K=20
basis=legendre   K=25
basis=legendre   K=30
basis=legendre   K=35
basis=legendre   K=40
basis=legendre   K=45
basis=legendre   K=50
basis=bspline    K=05
basis=bspline    K=10
basis=bspline    K=15
basis=bspline    K=20
basis=bspline    K=25
basis=bspline    K=30
basis=bspline    K=35
basis=bspline    K=40
basis=bspline    K=45
basis=bspline    K=50
basis=fourier    K=05
basis=fourier    K=10
basis=fourier    K=15
basis=fourier    K=20
basis=fourier    K=25
basis=fourier    K=30
basis=fourier    K=35
basis=fourier    K=40
basis=fourier    K=45
basis=fourier    K=50
Generated feature files: 40
Per-star metrics: /Users/erikak/Documents/uni/bakalauras/kodas/bp_rp_basis_experiment/result

## Basis-only evaluation first (`smoothing = 'none'`)

In [21]:
basis_only_combined = step02.rank_basis_only_configs(summary, arm="bp_rp_combined", smoothing="none")
basis_only_best_per_basis = step02.best_k_per_basis(basis_only_combined)

_hide_smoothed = lambda df: df[[c for c in df.columns if "_to_smoothed" not in c and c != "index"]]

print("Best no-smoothing K inside each basis family:")
display(_hide_smoothed(basis_only_best_per_basis))

print("Top no-smoothing BP+RP combined configurations overall:")
display(_hide_smoothed(basis_only_combined.head(15)))

Best no-smoothing K inside each basis family:


,basis,arm,smoothing,n_coeffs,rmse_to_raw_mean,rmse_to_raw_std,mae_to_raw_mean,mae_to_raw_std,rel_l2_to_raw_mean,rel_l2_to_raw_std,r2_to_raw_mean,r2_to_raw_std
0,bspline,bp_rp_combined,none,50,0.001503,0.000909,0.000653,0.000374,0.019796,0.011978,0.997202,0.004347
1,chebyshev,bp_rp_combined,none,50,0.001524,0.000921,0.000907,0.000535,0.020072,0.012129,0.997129,0.004339
2,legendre,bp_rp_combined,none,50,0.001524,0.000921,0.000907,0.000535,0.020072,0.012129,0.997129,0.004339
3,fourier,bp_rp_combined,none,50,0.007719,0.001206,0.002478,0.000480,0.101679,0.015886,0.948754,0.011034


Top no-smoothing BP+RP combined configurations overall:


,arm,basis,smoothing,n_coeffs,rmse_to_raw_mean,rmse_to_raw_std,mae_to_raw_mean,mae_to_raw_std,rel_l2_to_raw_mean,rel_l2_to_raw_std,r2_to_raw_mean,r2_to_raw_std
0,bp_rp_combined,bspline,none,50,0.001503,0.000909,0.000653,0.000374,0.019796,0.011978,0.997202,0.004347
1,bp_rp_combined,chebyshev,none,50,0.001524,0.000921,0.000907,0.000535,0.020072,0.012129,0.997129,0.004339
2,bp_rp_combined,legendre,none,50,0.001524,0.000921,0.000907,0.000535,0.020072,0.012129,0.997129,0.004339
3,bp_rp_combined,bspline,none,45,0.001785,0.001038,0.000851,0.000482,0.023515,0.013674,0.996145,0.005689
4,bp_rp_combined,chebyshev,none,45,0.001814,0.001019,0.001107,0.000618,0.023894,0.013425,0.996083,0.005563
5,bp_rp_combined,legendre,none,45,0.001814,0.001019,0.001107,0.000618,0.023894,0.013425,0.996083,0.005563
6,bp_rp_combined,bspline,none,40,0.002034,0.001128,0.001048,0.000586,0.026797,0.014855,0.995116,0.006772
7,bp_rp_combined,chebyshev,none,40,0.002086,0.001110,0.001293,0.000689,0.027474,0.014622,0.994954,0.006803
8,bp_rp_combined,legendre,none,40,0.002086,0.001110,0.001293,0.000689,0.027474,0.014622,0.994954,0.006803
9,bp_rp_combined,bspline,none,35,0.002247,0.001219,0.001250,0.000687,0.029593,0.016059,0.994111,0.007931


## BP+RP combined ranking

In [19]:
_hide_smoothed = lambda df: df[[c for c in df.columns if "_to_smoothed" not in c and c != "index"]]
_hide_smoothed(combined_ranking.head(15))

,arm,basis,smoothing,n_coeffs,rmse_to_raw_mean,rmse_to_raw_std,mae_to_raw_mean,mae_to_raw_std,rel_l2_to_raw_mean,rel_l2_to_raw_std,r2_to_raw_mean,r2_to_raw_std
0,bp_rp_combined,bspline,none,50,0.001503,0.000909,0.000653,0.000374,0.019796,0.011978,0.997202,0.004347
1,bp_rp_combined,chebyshev,none,50,0.001524,0.000921,0.000907,0.000535,0.020072,0.012129,0.997129,0.004339
2,bp_rp_combined,legendre,none,50,0.001524,0.000921,0.000907,0.000535,0.020072,0.012129,0.997129,0.004339
3,bp_rp_combined,bspline,none,45,0.001785,0.001038,0.000851,0.000482,0.023515,0.013674,0.996145,0.005689
4,bp_rp_combined,chebyshev,none,45,0.001814,0.001019,0.001107,0.000618,0.023894,0.013425,0.996083,0.005563
5,bp_rp_combined,legendre,none,45,0.001814,0.001019,0.001107,0.000618,0.023894,0.013425,0.996083,0.005563
6,bp_rp_combined,bspline,none,40,0.002034,0.001128,0.001048,0.000586,0.026797,0.014855,0.995116,0.006772
7,bp_rp_combined,chebyshev,none,40,0.002086,0.001110,0.001293,0.000689,0.027474,0.014622,0.994954,0.006803
8,bp_rp_combined,legendre,none,40,0.002086,0.001110,0.001293,0.000689,0.027474,0.014622,0.994954,0.006803
9,bp_rp_combined,bspline,none,35,0.002247,0.001219,0.001250,0.000687,0.029593,0.016059,0.994111,0.007931


## Star-level check after basis choice

In [23]:
chosen_basis = str(basis_only_best_per_basis.iloc[0]["basis"])
chosen_source_id = selected_source_id
chosen_k_list = basis_only_combined[basis_only_combined["basis"] == chosen_basis]["n_coeffs"].head(4).tolist()
if not chosen_k_list:
    chosen_k_list = [5, 10, 20, 40]

fig = step02.plot_selected_star_reconstructions_interactive(
    bp,
    rp,
    source_id=chosen_source_id,
    bases=[chosen_basis],
    k_list=chosen_k_list,
    smoothing="none",
)
fig.show()